In [33]:
#PANDAS
import pandas as pd

#NUMPY
import numpy as np

#SAVE
import joblib

#SKLEARN
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier

#WARNINGS
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [34]:
# ==================== LOAD DATA ====================
df = pd.read_csv('diabetes.csv')
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [35]:
df.shape

(768, 9)

In [36]:
# ==================== ADVANCED DATA CLEANING ====================
df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']] = df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']].replace(0, np.nan)

In [37]:
# Use KNN Imputer instead of Median (This boosts accuracy)
imputer = KNNImputer(n_neighbors=5)
df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']] = imputer.fit_transform(df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']])

In [38]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [39]:
# ==================== ADVANCED FEATURE ENGINEERING ====================
df['Insulin_Log'] = np.log1p(df['Insulin'])
df['DPF_Log'] = np.log1p(df['DiabetesPedigreeFunction'])

df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Age_BMI'] = df['Age'] * df['BMI']
df['Insulin_Glucose'] = df['Insulin'] / (df['Glucose'] + 1)
df['Glucose_Age'] = df['Glucose'] * df['Age']

In [40]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Insulin_Log,DPF_Log,Glucose_BMI,Age_BMI,Insulin_Glucose,Glucose_Age
0,6,148.0,72.0,35.0,209.0,33.6,0.627,50,1,5.347108,0.486738,4972.8,1680.0,1.402685,7400.0
1,1,85.0,66.0,29.0,73.0,26.6,0.351,31,0,4.304065,0.300845,2261.0,824.6,0.848837,2635.0
2,8,183.0,64.0,29.4,383.2,23.3,0.672,32,1,5.951163,0.514021,4263.9,745.6,2.082609,5856.0
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0,4.553877,0.154436,2500.9,590.1,1.044444,1869.0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1,5.129899,1.190279,5904.7,1422.3,1.217391,4521.0


In [41]:
# SPLITTING THE DATA
x = df.drop(columns=['Outcome'])
y = df['Outcome']

In [42]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [43]:
# HANDLING IMBALANCE
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [44]:
print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_resampled).value_counts().to_dict()}")

Before SMOTE: {0: 400, 1: 214}
After SMOTE: {0: 400, 1: 400}


In [45]:
# SCALING THE DATA
scaled = StandardScaler()

In [46]:
x_train_scaled = scaled.fit_transform(x_train_resampled)
x_test_scaled = scaled.transform(x_test)

In [47]:
# ==================== ENSEMBLE MODEL (VOTING CLASSIFIER) ====================
model1 = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8, 
    random_state=42, eval_metric='logloss', use_label_encoder=False
)

In [48]:
model2 = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_split=2, 
    random_state=42, n_jobs=-1
)

In [49]:
model3 = LogisticRegression(max_iter=1000, random_state=42)

In [50]:
ensemble_model = VotingClassifier(
    estimators=[('xgb', model1), ('rf', model2), ('lr', model3)],
    voting='soft'
)

In [51]:
# TRAIN
ensemble_model.fit(x_train_scaled, y_train_resampled)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('xgb', ...), ('rf', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[XGBClassifier...ree=None, ...), RandomForestC...ndom_state=42), LogisticRegre...ndom_state=42)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,14


In [52]:
# ==================== THRESHOLD TUNING ====================
y_proba = ensemble_model.predict_proba(x_test_scaled)[:, 1]

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.3, 0.7, 0.01):
    y_pred_temp = (y_proba >= threshold).astype(int)
    f1_temp = f1_score(y_test, y_pred_temp)
    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

In [53]:
print(f"\n🎯 Best Threshold: {best_threshold:.2f}")

y_pred = (y_proba >= best_threshold).astype(int)

print("\n" + "="*50)
print("MODEL PERFORMANCE ON TEST SET")
print("="*50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")


🎯 Best Threshold: 0.41

MODEL PERFORMANCE ON TEST SET
Accuracy:  0.7597
Precision: 0.6164
Recall:    0.8333
F1-Score:  0.7087
ROC-AUC:   0.8270


In [54]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))


Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.89      0.72      0.80       100
    Diabetes       0.62      0.83      0.71        54

    accuracy                           0.76       154
   macro avg       0.75      0.78      0.75       154
weighted avg       0.79      0.76      0.77       154



In [55]:
# SAVE MODEL (Saves the imputer now!)
joblib.dump(ensemble_model, 'diabetes_model.pkl')
joblib.dump(scaled, 'scaler.pkl')
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(x.columns.tolist(), 'feature_columns.pkl')
joblib.dump(best_threshold, 'threshold.pkl')

print("\n✅ All 5 files saved successfully!")


✅ All 5 files saved successfully!
